In [1]:
!pip install pandas
!pip install torch
!pip install transformers
!pip install scikit-learn
!pip install numpy
!pip install urllib3
!pip install onnxruntime
!pip install onnx
!pip install thread6

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 41.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 30.7 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
import torch
import numpy as np
import pandas as pd
import urllib.request
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup, BertModel

# Download and process data
url = "<dataset-url>"  # dataset CSV location removed; set before running

#download and read the contents
headers = {'User-Agent': 'Mozilla/5.0'}
request = urllib.request.Request(url, headers=headers)
df = pd.read_csv(urllib.request.urlopen(request), encoding='utf-16')

# Define sliding window parameters
max_seq_length = 512
stride = 128  # Overlap between consecutive windows

# Initialize tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')

def process_text(text):
  """
  Splits a long text into overlapping windows and tokenizes them.
  """
  windows = []
  for i in range(0, len(text), stride):
    window = text[i:i+max_seq_length]
    # Truncate if necessary
    if len(window) > max_seq_length:
      window = window[:max_seq_length]
    encoded_window = tokenizer.encode(window, add_special_tokens=True, padding='max_length', truncation=True)
    windows.append(encoded_window)
  return windows

# Process text data with sliding window
processed_data = []
for text, label in zip(df['hex'], df['label']):
  windows = process_text(text)
  for window in windows:
    processed_data.append([window, label])

# Convert to tensors and create datasets and loaders
input_ids, labels = zip(*processed_data)
input_ids = torch.tensor(input_ids)
attention_mask = torch.where(input_ids != 0, 1, 0)
labels = torch.tensor(labels)
dataset = TensorDataset(input_ids, attention_mask, labels)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

# Define model, optimizer, and loss function
model = BertModel.from_pretrained('bert-base-chinese')
model.pooler = torch.nn.Linear(model.config.hidden_size, 2)  # 2 classes for binary classification
optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = torch.nn.BCEWithLogitsLoss()

# Train the model
for epoch in range(5):
  for batch in train_loader:
    input_ids, attention_mask, labels = batch
    # Clear gradients
    optimizer.zero_grad()
    # Forward pass
    outputs = model(input_ids, attention_mask)
    logits = outputs.logits
    # Calculate loss
    loss = criterion(logits, labels.float())
    # Backpropagation
    loss.backward()
    # Update model weights
    optimizer.step()

# Evaluate the model
model.eval()
with torch.no_grad():
  val_predictions = []
  val_labels = []
  for batch in val_loader:
    input_ids, attention_mask, labels = batch
    outputs = model(input_ids, attention_mask)
    logits = outputs.logits
    val_predictions.extend(torch.sigmoid(logits).round())
    val_labels.extend(labels)
  val_accuracy = accuracy_score(val_predictions, val_labels)
  print("Validation Accuracy:", val_accuracy)

# Save the model in pt format
with open('/content/drive/My Drive/identifibertv10.pt', 'wb') as f:
    torch.save(model, f)


tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]